# 算法为什么会超时

[参考文章](https://programmercarl.com/algo/fundamentals/algorithm-timeout.html)

## 问题

算法结果正确，但运行时间超过题目限制，也会无法通过。判断能否完成时，需要同时考虑输入范围、操作次数和每次操作的成本

**不能把“每秒固定执行多少次 Python 操作”当作通用标准**，机器、数据类型、解释器和测试用例都会影响结果

## 思路

先估算数量级，再判断是否存在大量重复工作

| n | n 次操作 | $n^2$ 次操作 | $nlog_2 n$ 次操作，约数 |
| --- | --- | --- | --- |
| 1,000 | 1,000 | 1,000,000 | 10,000 |
| 100,000 | 100,000 | 10,000,000,000 | 1,660,000 |

这些是工作量估算，不是通过或超时的绝对界线。题目还有 q 次查询时，也要把 q 算进去

## 示例
### 重复求前缀的和

给定数组 nums，返回每个位置之前、包含当前位置的元素和。例如 [2,3,1] 返回 [2,5,6]


In [ ]:
def prefix_slow(nums: list[int]) -> list[int]:
    result = []

    for index in range(len(nums)):
        result.append(sum(nums[:index + 1]))

    return result


表面上只有一层循环，但每次 `nums[:index + 1]` 都会复制一个前缀，`sum()` 还要重新遍历它

时间复杂度：$O(n^2)$ 各次前缀的长度之和为 $1+2+cdots+n$

空间复杂度：$O(n)$ 包括结果列表和最多长度为 n 的临时切片

### 保存已有计算结果

用 total 保存当前前缀和，每次只加入一个新元素，避免反复复制、累加前面的内容


In [ ]:
def prefix_fast(nums: list[int]) -> list[int]:
    result = []
    total = 0

    for num in nums:
        total += num
        result.append(total)

    return result


时间复杂度：$O(n)$ 每个元素只累加一次，列表尾部追加按均摊 $O(1)$ 计算

空间复杂度：$O(n)$ 保存结果；如果不计返回结果，额外空间为 $O(1)$

## 小规模计时

`timeit.repeat()` 可以对同一份数据重复计时。下面仅用较小输入做观察，不把测试数据的创建时间算入算法时间


In [ ]:
from timeit import repeat

for n in (200, 400, 800):
    nums = list(range(n))
    slow_time = min(repeat(lambda: prefix_slow(nums), number=1, repeat=3))
    fast_time = min(repeat(lambda: prefix_fast(nums), number=1, repeat=3))
    print(f'n={n}: slow={slow_time:.6f}s, fast={fast_time:.6f}s')


计时结果会随环境变化，小输入还可能被函数调用等固定开销影响。看增长趋势比盯住某一次耗时更有意义，实验也不能代替复杂度推导

## 排查顺序

- 先检查是否写错了循环条件，导致指针不移动或无法结束
- 检查切片、求和、查找和排序是否被放进循环，产生重复工作
- 检查全部测试用例的总规模，以及递归是否反复计算同一个子问题
- 优先改进算法，再考虑减少无关输出等常数开销

[Python 文档：timeit](https://docs.python.org/3.11/library/timeit.html)
